# LangGraph Topic 3 — **Nodes**

**Official docs:** covered within the LangGraph Graph API / low-level graph concepts docs at [LangGraph docs](https://docs.langchain.com/oss/python/langgraph/graph-api?utm_source=chatgpt.com)

---

# 1) What is a Node in LangGraph?

A **node** is a **Python function that performs one unit of work inside the graph**.

In LangGraph, the graph does not run one big function from top to bottom.
Instead, it moves through **small functions (nodes)**.

Each node typically:

1. **receives the current state**
2. **does some work**
3. **returns updates to the state**

So if **State is the graph’s memory**, then **Nodes are the workers that operate on that memory**.

---

# 2) Mental Model

Think of a LangGraph app like a factory line:

* **State** = the shared box carrying data
* **Nodes** = workers on the line
* **Edges** = arrows telling which worker gets the box next

Example:

```text
User question
   ↓
[Node: rewrite_query]
   ↓
[Node: retrieve_docs]
   ↓
[Node: generate_answer]
   ↓
Final answer
```

Each node gets the current state, updates part of it, and hands it off.

---

# 3) Smallest Possible Node

```python
def hello_node(state):
    return {
        "answer": "Hello from LangGraph"
    }
```

This node:

* receives `state`
* ignores it in this example
* returns an update to the state

If current state is:

```python
{
    "question": "Hi",
    "answer": ""
}
```

after this node runs, state becomes:

```python
{
    "question": "Hi",
    "answer": "Hello from LangGraph"
}
```

---

# 4) Core Rule of Nodes

## A node is usually just a Python function.

Typical form:

```python
def my_node(state):
    ...
    return {...}
```

But LangGraph nodes can also accept additional runtime information:

```python
def my_node(state, config, runtime):
    ...
    return {...}
```

So conceptually a node can access:

* **state** → current graph data
* **config** → execution config
* **runtime** → runtime context / execution helpers

But when learning, start with:

```python
def my_node(state):
    ...
    return {...}
```

That’s the cleanest mental model.

---

# 5) What a Node Actually Does

A node can do **any Python work**.

For example, a node can:

* call an LLM
* rewrite a question
* classify a query
* search documents
* call a tool
* query a database
* validate data
* transform data
* choose a route
* increment counters
* format the final answer

So a node is **not tied to LLMs only**.
It’s just a step in the workflow.

---

# 6) Node = “Read State → Compute → Return Update”

This is the most important execution pattern.

```text
Current State
      ↓
Node reads state
      ↓
Node computes
      ↓
Node returns update
      ↓
LangGraph merges update into state
```

That’s the heart of node execution.

---

# 7) First Proper Example

## State

```python
from typing_extensions import TypedDict

class State(TypedDict):
    message: str
    reply: str
```

## Node

```python
def chatbot(state: State):
    user_msg = state["message"]

    return {
        "reply": f"Echo: {user_msg}"
    }
```

### What happens?

Input state:

```python
{
    "message": "Hello",
    "reply": ""
}
```

Node reads:

```python
state["message"]   # "Hello"
```

Node returns:

```python
{
    "reply": "Echo: Hello"
}
```

LangGraph merges that into state:

```python
{
    "message": "Hello",
    "reply": "Echo: Hello"
}
```

---

# 8) Nodes Usually Return Only the Updated Keys

If your state is:

```python
class State(TypedDict):
    question: str
    answer: str
    step_count: int
```

and your node only updates `answer`, then return only that:

```python
def answer_node(state: State):
    return {
        "answer": "This is the answer"
    }
```

You usually **do not** need to return:

```python
def answer_node(state: State):
    return {
        "question": state["question"],
        "answer": "This is the answer",
        "step_count": state["step_count"]
    }
```

That is redundant in most cases.

---

# 9) Node Input and Output: Very Clear Picture

## Input to node

The **current graph state**

## Output from node

A **dictionary of state updates**

So a node is not usually returning “the final result of the app.”
It is returning **a patch/update to the graph state**.

In [2]:
# Minimal Working Graph with One Node
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    text: str 
    result: str 

def process_text(state:State):
    return{
        "result": state['text'].upper()
    }

builder = StateGraph(State)
builder.add_node("process_text", process_text)
builder.add_edge(START, "process_text")
builder.add_edge("process_text", END)

graph = builder.compile()

output = graph.invoke({
    "text": "langgraph",
    "result": ""
})

print(output)

{'text': 'langgraph', 'result': 'LANGGRAPH'}


# 11) Real Node Flow Step by Step

Let’s trace the node execution.

## Initial state

```python
{
    "text": "langgraph",
    "result": ""
}
```

## Node runs

```python
def process_text(state: State):
    return {
        "result": state["text"].upper()
    }
```

It reads:

```python
state["text"]  # "langgraph"
```

It returns:

```python
{
    "result": "LANGGRAPH"
}
```

## State after merge

```python
{
    "text": "langgraph",
    "result": "LANGGRAPH"
}
```

That’s the full node lifecycle.

---

# 12) Nodes Can Read Values Produced by Earlier Nodes

This is how graphs become multi-step workflows.

Example state:

```python
class State(TypedDict):
    question: str
    search_query: str
    answer: str
```

## Node 1 — rewrite query

```python
def rewrite_query(state: State):
    return {
        "search_query": f"best explanation for {state['question']}"
    }
```

## Node 2 — generate answer

```python
def generate_answer(state: State):
    return {
        "answer": f"Using query: {state['search_query']}"
    }
```

Flow:

```text
START → rewrite_query → generate_answer → END
```

Here:

* Node 1 writes `search_query`
* Node 2 reads `search_query`

That is how nodes communicate: **through shared state**.

---

# 13) Types of Work a Node Can Perform

Let’s make this concrete.

---

## A) Transformation node

Turns one state value into another.

```python
def normalize_text(state):
    return {
        "clean_text": state["text"].strip().lower()
    }
```

---

## B) Classification node

```python
def classify_question(state):
    q = state["question"].lower()

    if "python" in q:
        return {"category": "programming"}
    return {"category": "general"}
```

---

## C) Tool-calling node

```python
def fake_tool_node(state):
    query = state["question"]
    tool_output = f"Tool result for: {query}"

    return {
        "tool_result": tool_output
    }
```

---

## D) LLM node

Pseudo-example:

```python
def llm_node(state):
    question = state["question"]

    # imagine an LLM call here
    response = f"LLM answer to: {question}"

    return {
        "answer": response
    }
```

---

## E) Counter / control node

```python
def increment_step(state):
    return {
        "step_count": state["step_count"] + 1
    }
```

---

# 14) Nodes Can Be Pure or Side-Effecting

## Pure-style node

Only reads state and returns updates.

```python
def summarize(state):
    return {"summary": state["text"][:20]}
```

## Side-effecting node

Calls external systems like:

* APIs
* databases
* LLMs
* tools
* search systems

```python
def call_search_api(state):
    query = state["query"]
    results = f"Search results for {query}"   # pretend API call
    return {"results": results}
```

LangGraph supports both.
In practical AI workflows, many nodes are side-effecting because they call models/tools.

---

# 15) Node Names vs Node Functions

When you add a node to a graph, you give it a **graph name** and a **function**.

```python
builder.add_node("chatbot", chatbot)
```

Here:

* `"chatbot"` → node name inside the graph
* `chatbot` → Python function

This is important because edges use the **node name**:

```python
builder.add_edge(START, "chatbot")
builder.add_edge("chatbot", END)
```

So the graph routes using names, not raw function objects.

In [3]:
# Example with Two Nodes

from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    text: str
    word_count: str
    result: str

def count_words(state:State):
    count = len(state['text'].split())

    return {
        "word_count": count
    }

def make_result(state:State):
    return {
        "result": f"This text has {state['word_count']} words"
    }

builder = StateGraph(State)
builder.add_node("count_words", count_words)
builder.add_node("make_result", make_result)

builder.add_edge(START, "count_words")
builder.add_edge("count_words", "make_result")
builder.add_edge("make_result", END)

graph = builder.compile()

output = graph.invoke({
    "text": "LangGraph makes workflows easier",
    "word_count": 0,
    "result": ""
})

print(output)

{'text': 'LangGraph makes workflows easier', 'word_count': 4, 'result': 'This text has 4 words'}


# 17) Step-by-Step State Through Those Two Nodes

## Initial state

```python
{
    "text": "LangGraph makes workflows easier",
    "word_count": 0,
    "result": ""
}
```

## After `count_words`

Node returns:

```python
{
    "word_count": 4
}
```

Merged state:

```python
{
    "text": "LangGraph makes workflows easier",
    "word_count": 4,
    "result": ""
}
```

## After `make_result`

Node returns:

```python
{
    "result": "This text has 4 words"
}
```

Final state:

```python
{
    "text": "LangGraph makes workflows easier",
    "word_count": 4,
    "result": "This text has 4 words"
}
```

This shows exactly how nodes chain together.

---

# 18) Node Signature Variants

You’ll mostly see this:

## Variant 1 — only state

```python
def my_node(state):
    ...
    return {...}
```

---

## Variant 2 — state + config

```python
def my_node(state, config):
    ...
    return {...}
```

Use this when you need runtime configuration values.

---

## Variant 3 — state + runtime

```python
def my_node(state, runtime):
    ...
    return {...}
```

Use this when you need access to runtime context.

---

## Variant 4 — state + config + runtime

```python
def my_node(state, config, runtime):
    ...
    return {...}
```

This is the most flexible form.

For learning, **start with only `state`**.

---

# 19) What is `config` in a Node?

`config` carries execution-specific settings.

Typical uses:

* thread IDs
* tracing metadata
* configurable runtime parameters
* execution options

You won’t need it in every beginner graph, but it becomes useful in production systems.

Conceptually:

```python
def my_node(state, config):
    print(config)
    return {...}
```

---

# 20) What is `runtime` in a Node?

`runtime` gives access to execution-time resources and context.

Examples of what runtime-related information may include:

* context passed to the graph
* stores / persistence helpers
* streaming / execution info
* runtime utilities

Example mental model:

```python
def my_node(state, runtime):
    provider = runtime.context.get("provider", "unknown")
    return {
        "answer": f"Provider used: {provider}"
    }
```

So:

* **state** = graph data
* **runtime** = execution support/context

---

# 21) State vs Node Responsibility

This is an important design separation.

## State answers:

**“What data is being carried through the graph?”**

## Node answers:

**“What work should happen at this step?”**

Example:

### State

```python
class State(TypedDict):
    question: str
    answer: str
    category: str
```

### Nodes

* `classify_question` → decides category
* `answer_question` → generates answer

State stores the data.
Nodes perform the work.

---

# 22) Good Node Design = One Clear Responsibility

A node should ideally do **one logical job**.

### Good design

```text
rewrite_query
retrieve_docs
generate_answer
format_output
```

Each node has one responsibility.

---

### Bad design

```text
do_everything_node
```

that:

* rewrites the question
* retrieves docs
* calls tools
* generates answer
* formats output
* updates counters

This makes graphs hard to debug and hard to reuse.

---

# 23) Practical Node Design Guideline

Try to make a node represent **one meaningful workflow step**.

Examples of good node boundaries:

* `rewrite_query`
* `classify_intent`
* `search_docs`
* `call_tool`
* `generate_answer`
* `check_if_done`
* `summarize_results`

This makes the graph readable and modular.

---

# 24) Nodes Can Also Route Using `Command`

Normally, a node just returns a state update.

But a node can also return a **Command** to:

* update state
* decide where to go next

Example conceptually:

```python
from langgraph.types import Command

def router_node(state):
    if "tool" in state["question"].lower():
        return Command(
            update={"needs_tool": True},
            goto="tool_node"
        )

    return Command(
        update={"needs_tool": False},
        goto="answer_node"
    )
```

So nodes can be both:

* **workers**
* **decision points**

We’ll revisit this more when we cover **Edges** and routing.

---

# 25) Nodes and Errors

A node is just Python code, so normal Python errors can happen.

Example:

```python
def bad_node(state):
    return {
        "count": 10 / 0
    }
```

This raises an error.

So in real apps, nodes often need:

* validation
* try/except around external API calls
* fallback logic
* retry strategy

Example:

```python
def safe_node(state):
    try:
        value = 10 / 2
        return {"result": value}
    except Exception as e:
        return {"error": str(e)}
```

---

# 26) Nodes and Shared State Communication

Nodes don’t “call each other” directly in the normal sense.

Instead, communication happens like this:

```text
Node A writes to state
        ↓
Node B reads from state
```

Example:

```python
def node_a(state):
    return {"query": "python tutorial"}

def node_b(state):
    return {"answer": f"Searching for {state['query']}"}
```

So the **state is the communication channel** between nodes.

---

# 27) Example: Mini RAG-Style Node Workflow

## State

```python
class State(TypedDict):
    question: str
    rewritten_query: str
    docs: list[str]
    answer: str
```

## Node 1 — rewrite query

```python
def rewrite_query(state: State):
    return {
        "rewritten_query": f"Detailed search for: {state['question']}"
    }
```

## Node 2 — retrieve docs

```python
def retrieve_docs(state: State):
    q = state["rewritten_query"]
    docs = [f"Doc about {q}", f"Another doc about {q}"]
    return {
        "docs": docs
    }
```

## Node 3 — answer

```python
def answer_question(state: State):
    docs_text = " | ".join(state["docs"])
    return {
        "answer": f"Answer based on docs: {docs_text}"
    }
```

Flow:

```text
START → rewrite_query → retrieve_docs → answer_question → END
```

This is a clean example of how nodes build an AI workflow step by step.

---

# 28) Nodes in Real LangGraph Systems

In real projects, common node types include:

## Input processing nodes

* clean input
* classify input
* rewrite query

## Retrieval nodes

* search vector DB
* search documents
* fetch records

## Tool nodes

* calculator tool
* weather API
* database tool
* web search tool

## LLM nodes

* generate answer
* summarize docs
* extract structured info
* critique previous answer

## Control nodes

* decide whether tool use is needed
* check retry limits
* branch to fallback flow

## Human nodes / interrupt points

* ask for approval
* pause for correction
* wait for user confirmation

---

# 29) Common Beginner Mistakes with Nodes

## Mistake 1: treating a node like a full standalone app

A node should usually be one step, not the whole system.

---

## Mistake 2: returning values not aligned with the state schema

If state is:

```python
class State(TypedDict):
    question: str
    answer: str
```

and node returns:

```python
{"result": "hello"}
```

that’s a bad schema mismatch if your graph expects `answer`.

---

## Mistake 3: putting too much into one node

If a node is doing five unrelated jobs, the graph becomes hard to maintain.

---

## Mistake 4: forgetting that later nodes depend on earlier state updates

If Node B needs `search_query`, then Node A must write it correctly.

---

## Mistake 5: confusing state with runtime dependencies

Things like:

* model client
* DB connection
* API object

usually belong in **runtime context**, not in state.

---

# 30) Node Best Practices

## Best practice 1 — one job per node

Example:

* classify
* retrieve
* answer
* format

---

## Best practice 2 — use clear names

Good:

* `rewrite_query`
* `retrieve_docs`
* `generate_answer`
* `tool_router`

Less clear:

* `process`
* `do_work`
* `handle_stuff`

---

## Best practice 3 — return only the keys you update

---

## Best practice 4 — make node boundaries meaningful

Each node should correspond to a logical workflow step.

---

## Best practice 5 — keep node code readable

Nodes are the building blocks of your graph. Keep them small enough to understand quickly.

---

# 31) Interview-Level Summary of Nodes

If asked in an interview:

## **What is a node in LangGraph?**

A node is a Python function representing one step of work in the graph. It receives the current graph state, performs some computation such as calling an LLM, a tool, or Python logic, and returns updates to the state. Nodes are connected by edges to form the execution flow of the graph.

---

# 32) Quick Revision Notes

## Node in one line

**A node is a workflow step implemented as a Python function.**

## A node usually does

* read state
* compute
* return state updates

## Nodes can perform

* LLM calls
* tool calls
* routing
* retrieval
* transformations
* validation
* control logic

## Typical node signature

```python
def my_node(state):
    return {...}
```

## More advanced signature

```python
def my_node(state, config, runtime):
    return {...}
```

---

# 33) Full Cheat Sheet — Nodes

```text
NODES IN LANGGRAPH

Definition
----------
A node is a Python function that performs one step of work in the graph.

Node workflow
-------------
1. Receive current state
2. Read needed values
3. Perform work
4. Return state updates

What a node can do
------------------
- Call an LLM
- Call a tool
- Query a DB
- Rewrite a question
- Classify input
- Update counters
- Format output
- Decide next step

Typical signature
-----------------
def my_node(state):
    return {...}

Advanced signature
------------------
def my_node(state, config, runtime):
    return {...}

Best practices
--------------
- One logical responsibility per node
- Use clear names
- Return only updated keys
- Keep nodes modular
- Use state for data sharing between nodes
```

---

# 34) Mini Assignment for You — Nodes

Build a graph with this state:

```python
class State(TypedDict):
    text: str
    cleaned_text: str
    length: int
```

## Node 1 — `clean_text`

* convert `text` to lowercase
* remove leading/trailing spaces
* store in `cleaned_text`

## Node 2 — `count_length`

* read `cleaned_text`
* store its length in `length`

## Flow

```text
START → clean_text → count_length → END
```

Input:

```python
{
    "text": "   HELLO LangGraph   ",
    "cleaned_text": "",
    "length": 0
}
```

Expected final idea:

```python
{
    "text": "   HELLO LangGraph   ",
    "cleaned_text": "hello langgraph",
    "length": 15
}
```

In [4]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    text: str
    clean_text: str
    length: str

def clean_text(state: State):
    space_remove = state['text'].strip()
    clean_txt = space_remove.lower()
    return {
        "clean_text": clean_txt
    }

def count_length(state: State):
    return{
        "length": len(state['clean_text'])
    }


builder = StateGraph(State)
builder.add_node("clean_text", clean_text)
builder.add_node("count_length", count_length)

builder.add_edge(START, "clean_text")
builder.add_edge("clean_text", "count_length")
builder.add_edge("count_length", END)

graph = builder.compile()

output = graph.invoke({
    "text": "   HELLO LangGraph   ",
    "cleaned_text": "",
    "length": 0
})

print(output)

{'text': '   HELLO LangGraph   ', 'clean_text': 'hello langgraph', 'length': 15}


---

# 35) One Important Bridge to the Next Topic

Now that you understand **nodes**, the next question becomes:

## “How does LangGraph decide which node runs next?”

That is exactly the job of **Edges**.

So the flow of concepts is now:

```text
State   = what data moves through the graph
Nodes   = what work happens at each step
Edges   = how control moves from node to node
```
